# Member 1 - Data preparation
Downloads INCLUDE, selects the INCLUDE-50 classes, splits, extracts frames and runs EDA.
**A CPU session is enough** for everything here (save GPU quota for training).

Output: `processed/` folder (splits + frames). Member 3 adds `processed/keypoints/`.
Share it with the team as a Kaggle Dataset or a shared Drive folder.

## 0. Setup (run every new session)
Replace `YOUR-ORG` with your GitHub user/organisation and `BRANCH` with your branch.

In [ ]:
REPO = "https://github.com/YOUR-ORG/isl-word-recognition.git"
BRANCH = "develop"          # or your feature branch while developing
import os
if not os.path.exists("isl-word-recognition"):
    !git clone -q $REPO
%cd isl-word-recognition
!git fetch -q && git checkout -q $BRANCH && git pull -q
!git log --oneline -1

In [ ]:
# Colab: mount Google Drive so data and checkpoints survive disconnects.
# Kaggle: skip this cell (data goes to /kaggle/working, attach datasets via "Add data").
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab")

In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
SETUP_SCRIPT = "scripts/setup_kaggle.sh" if ON_KAGGLE else "scripts/setup_colab.sh"
!bash $SETUP_SCRIPT

**Where things are stored** (auto-detected, override with environment variables):

| | Colab (Drive mounted) | Kaggle |
|---|---|---|
| data | `/content/drive/MyDrive/isl_data` | `/kaggle/working/isl_data` |
| results | `/content/drive/MyDrive/isl_results` | `/kaggle/working/results` |

On Kaggle, if the processed data is attached as a dataset, point to it with
`--set paths.processed_dir=/kaggle/input/<dataset-name>` (or the `PROCESSED` variable below).

In [ ]:
PROCESSED = ""   # e.g. "/kaggle/input/isl-include50-processed"  (leave empty on Colab)
SET = f"paths.processed_dir={PROCESSED}" if PROCESSED else ""
print("extra overrides:", SET or "(none)")

## Quick check on synthetic data (1-2 minutes, CPU is fine)
Run this once before using GPU time on the real data.

In [ ]:
# Make a small fake dataset and run all tests: checks the setup before using real data.
!python scripts/make_dummy_data.py --data-root /tmp/isl_dummy --classes 6 --videos 8 --fake-keypoints
!python -m pytest -q

## 1. Download INCLUDE (one zip at a time, only the needed classes are extracted)
If you have the official INCLUDE-50 class list, save it as `configs/include50_classes.txt` first.

In [ ]:
!python -m src.data.download --list      # see the zip files and their sizes

In [ ]:
# Downloads each zip, extracts only the needed classes, deletes the zip. Safe to re-run after a disconnect.
!python -m src.data.download             # add --include "Adjectives*" to test with one zip

## 2. Split (writes `label_map.json`, train/val/test CSVs)

In [ ]:
# Scan the videos, pick the classes, split per class into train/val/test, save label_map.json.
# Then show the first rows of train.csv (Colab path first, Kaggle path if that fails).
!python -m src.data.split
!head -5 /content/drive/MyDrive/isl_data/processed/splits/train.csv 2>/dev/null || head -5 /kaggle/working/isl_data/processed/splits/train.csv

## 3. Extract frames for Pipeline A (~2-3 GB for INCLUDE-50 at 224px, 24 frames)

In [ ]:
# Save 24 resized frames per video as .npy files (input for Member 2's CNN+LSTM).
!python -m src.features.extract_frames --set frames.workers=2

## 4. EDA

In [ ]:
# Dataset statistics (videos per class, video length, ...) and show the saved plots.
!python -m src.data.eda
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("/content/drive/MyDrive/isl_results/eda/*.png") + glob.glob("/kaggle/working/results/eda/*.png")):
    display(Image(p))

## 5. Share the processed data
**Kaggle:** in the right panel open *Output* → *New Dataset* from `/kaggle/working/isl_data/processed`
(name it e.g. `isl-include50-processed`), and share it with your teammates.

**Colab:** share the Drive folder `isl_data/processed` with your teammates; each of them adds a
shortcut to it in their own *My Drive* so the path is the same.